# Avaliação de cenários hipotéticos para sequestro de carbono

Este notebook usa a [API COMET-Farm](https://gitlab.com/comet-api/api-docs/-/tree/master/) para derivar informações sobre o sequestro de carbono no solo para campos agrícolas.

### Configuração do ambiente Micromamba

Para instalar os pacotes necessários, consulte nossa [documentação](https://github.com/microsoft/farmvibes-ai/tree/main/notebooks).

### Dados de entrada Necessários

O FarmVibes.AI depende dos objetos [`SeasonalFieldInformation`](https://github.com/microsoft/farmvibes-ai/tree/main/src/vibe_core/vibe_core/data/farm.py) para representar as práticas agrícolas em um campo durante uma determinada estação. Esses objetos contêm informações relacionadas à colheita, plantio, preparo do solo, adubação orgânica e aplicação de fertilizantes. Esses objetos contêm informações relacionadas à colheita, plantio, preparo do solo, adubação orgânica e aplicação de fertilizantes. Para estimar a quantidade de carbono sequestrado pelo solo de um determinado campo, precisamos fornecer duas listas de objetos `SeasonalFieldInformation`:

 * `baseline_seasonal_fields: List[SeasonalFieldInformation]`: Contém informações sobre as atividades agrícolas passadas em um campo específico sob uma determinada gestão. Cada objeto `SeasonalFieldInformation` que representa um ano de informações de cultura e é recomendado usar **dez anos calendários** para representar as informações de carbono da linha de base.
 * `scenario_seasonal_fields: List[SeasonalFieldInformation]`: Contém informações sobre as atividades agrícolas a serem realizadas nos próximos **dois anos calendários**, após os campos definidos em `baseline_seasonal_fields`. Semelhante aos `baseline_seasonal_fields`, cada objeto `SeasonalFieldInformation` armazena informações de produção para cada ano da cultura.

Os objetos são armazenados nos arquivos [`baseline_seasonal_fields.json`](./baseline_seasonal_fields.json) e [`scenario_seasonal_fields.json`](./scenario_seasonal_fields.json) no formato JSON e são carregados na memória nas células seguintes.

### Importar os pacotes necessários

In [ ]:
import json
from datetime import datetime
from typing import Any, Dict

from vibe_core.data import (
    FertilizerInformation,
    HarvestInformation,
    OrganicAmendmentInformation,
    SeasonalFieldInformation,
    TillageInformation,
)

In [ ]:
def seasonal_field_from_dict(data: Dict[str, Any]) -> SeasonalFieldInformation:
    harvests = [HarvestInformation(**h) for h in data["harvests"]]
    tillages = [TillageInformation(**t) for t in data["tillages"]]
    organic_amendments = [OrganicAmendmentInformation(**o) for o in data["organic_amendments"]]
    fertilizers = [FertilizerInformation(**f) for f in data["fertilizers"]]

    return SeasonalFieldInformation(
        id=data["id"],
        time_range=(
            datetime.fromisoformat(data["time_range"][0]),
            datetime.fromisoformat(data["time_range"][1]),
        ),
        geometry=data["geometry"],
        assets=data["assets"],
        crop_name=data["crop_name"],
        crop_type=data["crop_type"],
        properties=data["properties"],
        fertilizers=fertilizers,
        harvests=harvests,
        tillages=tillages,
        organic_amendments=organic_amendments,
    )


### Carregar objetos SeasonFieldInformationObjects de FarmVibes.AI

A próxima célula carrega duas listas de objetos de campo sazonais, armazenados nos arquivos [`baseline_seasonal_fields.json`](./baseline_seasonal_fields.json) e [`scenario_seasonal_fields.json`](./scenario_seasonal_fields.json), que são necessários para a estimativa da sequestração de carbono no solo.

In [ ]:
baseline_fields_file = "./baseline_seasonal_fields.json"
with open(baseline_fields_file) as json_file:
    baseline_seasonal_fields = [
        seasonal_field_from_dict(seasonal_field_dict)
        for seasonal_field_dict in json.load(json_file)
    ]

scenario_fields_file = "./scenario_seasonal_fields.json"
with open(scenario_fields_file) as json_file:
    scenario_seasonal_fields = [
        seasonal_field_from_dict(seasonal_field_dict)
        for seasonal_field_dict in json.load(json_file)

]

## Detalhes sobre a saída da API COMET-Farm

1. Todos os modelos de gases de efeito estufa na plataforma COMET-Farm (Exemplo: DayCent, metano do arroz, queima de resíduos, calcário, fertilizante ureia, etc.) são executados contra o cenário base e, em seguida, contra cada cenário de conservação, para cada combinação única de unidades de mapa de solo encontrada dentro de cada parcela ou ponto, para cada modelo.

2. Também são retornados os totais agregados dos resultados "Baseline" e "Scenario" para todos os modelos, nomeados como tal.

Um exemplo de saída é o seguinte:

```json
{
    "@name": "cenário: 21/07/2022 10:34:05",
    "Carbono": {
        "Carbono no Solo": "-1679.9",
        "Carbono da Queima de Biomassa": "0",
        "Carbono do Solo (2000)": "5511.312",
        "Carbono do Solo (Início)": "5753.6314",
        "Carbono do Solo (Fim)": "5759.8725"
    },
    "CO2": {
        "CO2 da Calcário": "0",
        "CO2 da Fertilização com Ureia": "8.5587",
        "CO2 de Solos Orgânicos Drenados": "0"
    },
    "N2O": {
        "N2O no Solo": "536.1286",
        "N2O no Solo (Direto)": "451.9349",
        "N2O no Solo (Indireto - Evaporação)": "84.1937",
        "N2O no Solo (Indireto - Lixiviação)": "0",
        "Cultivo de Arroz em Pântano (N2O)": "0",
        "N2O da Queima de Biomassa": "0",
        "N2O de Solos Orgânicos Drenados": "0"
    },
    "CH4": {
        "CH4 no Solo": "0",
        "CH4 no Cultivo de Arroz em Pântano": "0",
        "CH4 da Queima de Biomassa": "0"
    }
}
```

## Requisitos para executar este notebook

1. Cadastre-se em https://comet-farm.com/. O endereço de e-mail registrado lá será usado quando houver mensagens de erro ou quando uma solicitação falhar.
2. Cadastre-se em https://dashboard.ngrok.com/ para permitir que o endpoint de carbono TerraVibes seja acessível pelos webhooks da API COMET-Farm.
   1. Navegue até "Começar"/"Seu token de autenticação" e copie o token de autenticação
   2. Atualize o token de autenticação copiado na variável "NGROK_AUTH_TOKEN" na próxima célula

In [ ]:
COMET_REGISTERED_EMAIL = ""
NGROK_AUTH_TOKEN = ""

In [ ]:

from vibe_core.client import FarmvibesAiClient, get_default_vibe_client

### Sobre executar este fluxo de trabalho

*Observação*: Executar este fluxo de trabalho exporá um endpoint dentro do contêiner do worker FarmVibes.AI publicamente. Essa conexão existirá enquanto o fluxo de trabalho estiver em execução e será fechada assim que o fluxo de trabalho for concluído.

Este é um passo necessário para receber os resultados da API COMET-Farm. Uma configuração incorreta do ngrok fará com que o fluxo de trabalho falhe.

In [ ]:
client: FarmvibesAiClient = get_default_vibe_client()
WORKFLOW_NAME = "farm_ai/carbon_local/carbon_whatif"

In [ ]:
client.document_workflow(WORKFLOW_NAME)

In [ ]:
parameters = {
  "ngrok_token": NGROK_AUTH_TOKEN,
  "comet_support_email": COMET_REGISTERED_EMAIL
}

run = client.run(
    WORKFLOW_NAME,
    name="whatif_carbon_seq",
    input_data={
        "baseline_seasonal_fields": baseline_seasonal_fields,
        "scenario_seasonal_fields": scenario_seasonal_fields,
    },
    parameters=parameters
)
run.monitor()

Use as células abaixo para inspecionar o fluxo de trabalho em execução.

Dependendo da disponibilidade dos recursos da API COMET-Farm, os fluxos de trabalho levarão mais tempo para serem concluídos.

Se o fluxo de trabalho não for concluído dentro de 2 (duas) horas e o fluxo de trabalho ainda estiver em execução, utilize o suporte do COMET-Farm em appnrel@colostate.edu. Em caso de falha, informações de erro serão enviadas para o endereço de e-mail registrado no COMET.

In [ ]:
carbon_offset_info = run.output["carbon_output"][0]
print(f"Estimated carbon offset is {carbon_offset_info.carbon}")